In [ ]:
import pandas as pd
books_df = pd.read_csv(r"../data/books_enriched.csv")

In [ ]:
books_df = books_df.drop(columns=["average_rating", "ratings_count"])

In [ ]:
print(books_df.columns)

In [ ]:
for col in [
    "title",
    "subtitle",
    "authors",
    "categories_final",
    "description_final"
]:
    books_df[col] = (
        books_df[col]
        .fillna("")
        .str.strip()
    )

In [ ]:
books_df = books_df[
    books_df["language"].fillna("").str.lower().eq("en")
].copy()

books_df.reset_index(drop=True, inplace=True)

In [ ]:
books_df["title_clean"] = (
    books_df["title"]
    .str.lower()
    .str.strip()
)

In [ ]:
duplicates = books_df[
    books_df["title_clean"].duplicated(keep=False)
].sort_values("title_clean")

print(
    duplicates[
        ["title_clean", "authors", "google_id", "isbn_10", "isbn_13"]
    ].to_string(index=False)
)

In [ ]:
import re
import unicodedata

def clean_author(author):
    if pd.isna(author):
        return ""

    author = unicodedata.normalize("NFKC", str(author))
    author = author.lower()

    # Remove punctuation except commas, which separate authors
    author = re.sub(r"[^\w\s,]", "", author, flags=re.UNICODE)

    # Normalize spaces without removing author separators
    author = re.sub(r"\s+", " ", author).strip()

    # Clean spaces around commas
    author = re.sub(r"\s*,\s*", ", ", author)

    return author


def normalize_title(title):
    title = str(title)
    title = unicodedata.normalize("NFKC", title)
    title = title.lower()
    title = re.sub(r"\([^)]*\)", "", title)
    title = re.sub(r"[^\w\s]", " ", title, flags=re.UNICODE)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def clean_category(category):
    category = str(category)
    category = unicodedata.normalize("NFKC", category)
    category = category.lower().strip()
    category = re.sub(
        r"[^\w\s,/-]",
        " ",
        category,
        flags=re.UNICODE
    )
    category = re.sub(r"\s+", " ", category).strip()
    return category


def clean_description(description):
    if pd.isna(description):
        return ""
    description = str(description)
    description = unicodedata.normalize("NFKC", description)
    description = re.sub(r"<[^>]+>", " ", description)
    description = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        description
    )
    description = re.sub(r"\s+", " ", description)
    description = "".join(
        char for char in description
        if unicodedata.category(char) != "Cc"
        or char in "\n\t"
    )
    description = re.sub(r"\s+", " ", description).strip()
    return description

In [ ]:
books_df["author_clean"] = books_df["authors"].apply(clean_author)
books_df["title_clean"] = books_df["title"].apply(normalize_title)
books_df["category_clean"] = books_df["categories_final"].apply(clean_category)
books_df["clean_description"] = books_df["description_final"].apply(clean_description)

In [ ]:
# Identifying spelling Variations
all_authors = set()

for authors in books_df["authors"].dropna():
    for author in str(authors).split(","):
        author = clean_author(author)
        if author:
            all_authors.add(author)

print("Unique author strings:", len(all_authors))

In [ ]:
from collections import defaultdict

authors_by_surname = defaultdict(list)

for author in all_authors:
    surname = author.split()[-1]
    authors_by_surname[surname].append(author)

In [ ]:
from rapidfuzz import fuzz

possible_author_aliases = []

for surname, authors in authors_by_surname.items():

    if len(authors) < 2:
        continue

    for i in range(len(authors)):
        for j in range(i + 1, len(authors)):

            a = authors[i]
            b = authors[j]

            score = fuzz.token_set_ratio(a, b)

            if score >= 70:
                possible_author_aliases.append({
                    "surname": surname,
                    "author1": a,
                    "author2": b,
                    "score": score
                })

In [ ]:
author_alias_df = pd.DataFrame(possible_author_aliases)

author_alias_df.sort_values(
    "score",
    ascending=False
).head(10)

In [ ]:
HONORIFICS = {
    "mr",
    "mrs",
    "ms",
    "miss",
    "dr",
    "prof",
    "professor",
    "sir",
    "lady",
    "lord",
    "canon",
    "count",
    "rev",
    "reverend",
}

SUFFIXES = {
    "jr",
    "sr",
    "ii",
    "iii",
    "iv",
    "v",
    "md",
    "phd",
    "esq",
}

In [ ]:
def remove_author_titles(author):
    words = clean_author(author).split()

    words = [
        word for word in words
        if word not in HONORIFICS
        and word not in SUFFIXES
    ]

    return " ".join(words)

In [ ]:
def author_name_parts(author):
    author = remove_author_titles(author)

    if not author:
        return []

    return author.split()

In [ ]:
def authors_likely_same(author1, author2):
    a = author_name_parts(author1)
    b = author_name_parts(author2)

    if not a or not b:
        return False

    # Require the same surname
    if a[-1] != b[-1]:
        return False

    first_a = "".join(a[:-1])
    first_b = "".join(b[:-1])

    # Exact match
    if first_a == first_b:
        return True

    # Compare initials against full names
    # Example: "jrr" vs "johnronaldreuel"
    if first_a and first_b:
        if len(first_a) <= len(first_b):
            shorter, longer = first_a, first_b
        else:
            shorter, longer = first_b, first_a

        if len(shorter) == 1:
            return longer.startswith(shorter)

        # Combined initials match the first letters of full-name components
        shorter_parts = a[:-1] if len(first_a) <= len(first_b) else b[:-1]
        longer_parts = b[:-1] if len(first_a) <= len(first_b) else a[:-1]

        if all(len(part) == 1 for part in shorter_parts):
            if len(shorter_parts) <= len(longer_parts):
                return all(
                    initial == full_name[0]
                    for initial, full_name in zip(
                        shorter_parts, longer_parts
                    )
                )

        # Combined initials, e.g. "jrr", against full-name initials
        initials = "".join(part[0] for part in longer_parts if part)
        if shorter == initials[:len(shorter)]:
            return True

    return False

In [ ]:
author_groups = []
assigned = set()

for surname, authors in authors_by_surname.items():

    for author in authors:
        if author in assigned:
            continue

        group = {author}

        for other in authors:
            if other == author or other in assigned:
                continue

            if authors_likely_same(author, other):
                group.add(other)

        assigned.update(group)
        author_groups.append(group)

In [ ]:
canonical_map = {}

for group in author_groups:
    canonical_name = min(group, key=len)

    for author in group:
        canonical_map[author] = canonical_name

In [ ]:
for author, canonical in list(canonical_map.items()):
    if author != canonical:
        print(f"{author}  ->  {canonical}")

In [ ]:
def get_author_set(author_string):
    if pd.isna(author_string) or not str(author_string).strip():
        return set()

    authors = str(author_string).split(",")
    canonical_authors = set()

    for author in authors:
        author = clean_author(author)

        if not author:
            continue

        canonical_author = canonical_map.get(
            author,
            author
        )

        canonical_authors.add(canonical_author)

    return canonical_authors


books_df["author_set"] = books_df["authors"].apply(get_author_set)

In [ ]:
books_df.loc[
    books_df["authors"].str.contains(
        "tolkien", case=False, na=False
    ),
    ["title", "author_set"]
].head(20)

In [ ]:
books_df.loc[
    books_df["authors"].str.contains(
        "Simone Brioni", na=False
    ),
    ["authors", "author_set"]
].head()

In [ ]:
print(books_df[[
    "authors",
    "author_set"
]].tail(10))

In [ ]:
import re
import unicodedata
import pandas as pd
from collections import defaultdict
from rapidfuzz import fuzz

TITLE_THRESHOLD = 75
AUTHOR_THRESHOLD = 75
MISSING_AUTHOR_TITLE_THRESHOLD = 92

books = books_df.copy().reset_index(drop=True)

books["title_clean"] = books["title_clean"].fillna("").astype(str)
books["author_clean"] = books["author_clean"].fillna("").astype(str)
books["authors"] = books["authors"].fillna("").astype(str)


def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text).casefold())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text)).strip()


def author_similarity(set1, set2):
    if not set1 or not set2:
        return None

    names1 = list(set1)
    names2 = list(set2)

    matches = []
    for i, a in enumerate(names1):
        for j, b in enumerate(names2):
            score = fuzz.ratio(a, b)
            if score >= 85:
                matches.append((score, i, j))

    # Greedy one-to-one matching
    matched1, matched2 = set(), set()

    for score, i, j in sorted(matches, reverse=True):
        if i not in matched1 and j not in matched2:
            matched1.add(i)
            matched2.add(j)

    matched = len(matched1)

    # Measures how much the smaller set overlaps the larger one
    return matched / min(len(names1), len(names2)) * 100

possible_duplicates = []

for index1, index2 in sorted(candidate_pairs):
    row1 = books.iloc[index1]
    row2 = books.iloc[index2]

    title1 = normalize_text(row1["title_clean"])
    title2 = normalize_text(row2["title_clean"])

    title_score = fuzz.ratio(title1, title2)

    if title_score < TITLE_THRESHOLD:
        continue

    authors1 = row1["author_set"]
    authors2 = row2["author_set"]

    # Treat missing or title-like author values as unknown
    unknown1 = (
        not authors1
        or normalize_text(row1["author_clean"]) == title1
    )
    unknown2 = (
        not authors2
        or normalize_text(row2["author_clean"]) == title2
    )

    if unknown1 or unknown2:
        author_score = None
        is_duplicate = title_score >= MISSING_AUTHOR_TITLE_THRESHOLD
    else:
        author_score = author_similarity(authors1, authors2)
        is_duplicate = (
            author_score is not None
            and author_score >= AUTHOR_THRESHOLD
        )

    if is_duplicate:
        possible_duplicates.append({
            "index1": index1,
            "index2": index2,
            "title1": row1["title"],
            "title2": row2["title"],
            "title_score": title_score,
            "authors1": row1["authors"],
            "authors2": row2["authors"],
            "author_set1": authors1,
            "author_set2": authors2,
            "author_score": author_score
        })

possible_duplicates_df = pd.DataFrame(possible_duplicates)

if not possible_duplicates_df.empty:
    possible_duplicates_df = possible_duplicates_df.sort_values(
        ["title_score", "author_score"],
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)

print("Potential duplicate pairs:", len(possible_duplicates_df))
display(possible_duplicates_df.head(50))

In [ ]:
duplicates_df = pd.DataFrame(possible_duplicates)

duplicates_df

In [ ]:
duplicates_df.loc[
    duplicates_df["title1"].str.contains(
        "rebecca", case=False, na=False
    ),
    ["title2", "title1", "authors1", "authors2"]
].head(20) 

In [ ]:
important_columns = [
    "authors",
    "publisher",
    "published_date",
    "description_final",
    "categories_final",
    "isbn_10",
    "isbn_13",
]

# Make empty strings and whitespace-only strings count as missing
for col in important_columns:
    books_df[col] = books_df[col].replace(
        r"^\s*$", pd.NA, regex=True
    )
books_df["completeness"] = (
    books_df[important_columns]
    .notna()
    .sum(axis=1)
)

In [ ]:
rows_to_drop = set()

for _, row in duplicates_df.iterrows():

    index1 = row["index1"]
    index2 = row["index2"]

    score1 = books_df.loc[index1, "completeness"]
    score2 = books_df.loc[index2, "completeness"]

    if books_df.loc[index1, "authors"] == "" or books_df.loc[index2, "authors"] == "":
        if books_df.loc[index1, "authors"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)

    elif books_df.loc[index1, "categories_final"] == "" or books_df.loc[index2, "categories_final"] == "":
        if books_df.loc[index1, "categories_final"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)

    elif books_df.loc[index1, "description_final"] == "" or books_df.loc[index2, "description_final"] == "":
        if books_df.loc[index1, "description_final"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)
     
    else:
        if score1 >= score2:
            rows_to_drop.add(index2)
        else:
            rows_to_drop.add(index1)

In [ ]:
print("books to drop:" , len(rows_to_drop))

In [ ]:
books_df = books_df.drop(
    index=list(rows_to_drop)
).reset_index(drop=True)

In [ ]:
books_df.loc[
    books_df["title"].str.contains(
        "rebecca", case=False, na=False
    ),
    ["title", "author_clean"]
].head(20)

In [ ]:
NON_BOOK_TERMS = [
    "notebook",
    "workbook",
    "study guide",
    "student workbook",
    "teacher guide",
    "teacher's guide",
    "answer key",
    "summary",
    "study notes",
    "companion guide",
    "lesson plan",
    "test prep",
    "exam prep",
    "journal",
    "coloring book",
    "activity book"
]

In [ ]:
def is_non_book(title):
    title = normalize_title(title)

    for term in NON_BOOK_TERMS:
        if term in title:
            return True

    return False

In [ ]:
books_df = books_df[
    ~books_df["title"].apply(is_non_book)
].copy()

books_df = books_df[
    ~books_df["author_clean"].apply(is_non_book)
].copy()

books_df.reset_index(drop=True, inplace=True)

In [ ]:
books_df.head(10)

In [ ]:
books_df.to_csv(
    "../data/processed_books_dataset.csv",
    index=False
)

print("Saved:", len(books_df), "books")